# Evaluate Models

Notebook này đọc các file dự đoán từ `results/predictions/`, sau đó tính:

- Accuracy
- Macro-F1
- Macro-Precision
- Macro-Recall
- Weighted Precision/Recall/F1
- Classification report
- Confusion matrix
- Error analysis CSV và summary CSV cho notebook phân tích lỗi

File này chỉ chấm điểm và lưu artifact. Việc đọc lỗi chi tiết, chọn model tốt nhất để xem nhầm lẫn và vẽ biểu đồ nằm trong `notebook/04_error_analysis.ipynb`.

In [1]:
from pathlib import Path
import json

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

In [2]:
PREDICTIONS_DIR = Path("../results/predictions")
EVALUATION_DIR = Path("../results/evaluation")

EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

prediction_files = sorted(PREDICTIONS_DIR.glob("*_predictions.csv"))
if not prediction_files:
    raise FileNotFoundError("Chưa có file predictions. Hãy chạy src/train_baseline.ipynb trước.")

prediction_files

[PosixPath('../results/predictions/lightgbm_predictions.csv'),
 PosixPath('../results/predictions/linear_svc_predictions.csv'),
 PosixPath('../results/predictions/logistic_regression_predictions.csv'),
 PosixPath('../results/predictions/multinomial_nb_predictions.csv'),
 PosixPath('../results/predictions/random_forest_predictions.csv')]

## 1. Metric functions

In [3]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "weighted_precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "weighted_recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
    }


def evaluate_prediction_file(prediction_file):
    model_name = prediction_file.name.replace("_predictions.csv", "")
    model_dir = EVALUATION_DIR / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    pred_df = pd.read_csv(prediction_file).dropna(subset=["true_label", "predicted_label"])
    pred_df["true_label"] = pred_df["true_label"].astype(str)
    pred_df["predicted_label"] = pred_df["predicted_label"].astype(str)

    y_true = pred_df["true_label"]
    y_pred = pred_df["predicted_label"]
    labels = sorted(set(y_true) | set(y_pred))

    metrics = compute_metrics(y_true, y_pred)
    metrics.update({
        "model": model_name,
        "test_samples": int(len(pred_df)),
        "num_labels": int(len(labels)),
    })

    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report).transpose().to_csv(model_dir / "classification_report.csv")

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    pd.DataFrame(cm, index=labels, columns=labels).to_csv(model_dir / "confusion_matrix.csv")

    error_df = pred_df.copy()
    error_df["is_error"] = error_df["true_label"] != error_df["predicted_label"]
    error_df.to_csv(model_dir / "error_analysis.csv", index=False)

    misclassified = (
        error_df[error_df["is_error"]]
        .groupby(["true_label", "predicted_label"])
        .size()
        .reset_index(name="error_count")
        .sort_values("error_count", ascending=False)
    )
    misclassified.to_csv(model_dir / "misclassification_summary.csv", index=False)

    with open(model_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    return metrics

## 2. Run evaluation

In [4]:
Sall_metrics = []

for prediction_file in prediction_files:
    print(f"[*] Evaluating {prediction_file.name}")
    all_metrics.append(evaluate_prediction_file(prediction_file))

metrics_summary = pd.DataFrame(all_metrics).sort_values("macro_f1", ascending=False)
metrics_summary.to_csv(EVALUATION_DIR / "metrics_summary.csv", index=False)
metrics_summary

[*] Evaluating lightgbm_predictions.csv
[*] Evaluating linear_svc_predictions.csv
[*] Evaluating logistic_regression_predictions.csv
[*] Evaluating multinomial_nb_predictions.csv
[*] Evaluating random_forest_predictions.csv


,accuracy,macro_f1,macro_precision,macro_recall,weighted_f1,weighted_precision,weighted_recall,model,test_samples,num_labels
1,0.496392,0.364027,0.365065,0.393393,0.485601,0.497788,0.496392,linear_svc,2633,154
2,0.391948,0.309539,0.324122,0.376059,0.387314,0.490207,0.391948,logistic_regression,2633,154
4,0.424991,0.263771,0.343012,0.261170,0.381578,0.425266,0.424991,random_forest,2633,154
3,0.245727,0.031416,0.069758,0.039316,0.157106,0.239547,0.245727,multinomial_nb,2633,154
0,0.033802,0.017794,0.019373,0.024591,0.038134,0.056636,0.033802,lightgbm,2633,154


## 3. Output files

Sau khi chạy xong, notebook này tạo các file:

- `results/evaluation/metrics_summary.csv`
- `results/evaluation/<model>/metrics.json`
- `results/evaluation/<model>/classification_report.csv`
- `results/evaluation/<model>/confusion_matrix.csv`
- `results/evaluation/<model>/error_analysis.csv`
- `results/evaluation/<model>/misclassification_summary.csv`

Chạy tiếp `notebook/04_error_analysis.ipynb` để phân tích các file này.